In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install pillow opencv-python==4.8.0.76 pytesseract==0.3.10

In [ ]:
def pre_crop_top_middle(cv_img, debug_dir, namestem):
    """粗裁剪：只保留上1/3 + 中间1/3"""
    h, w = cv_img.shape[:2]
    top, bottom = 0, h // 3
    left, right = w // 3, 2 * w // 3
    cropped = cv_img[top:bottom, left:right]
    cv2.imwrite(str(Path(debug_dir) / f"{namestem}_precrop.jpg"), cropped)
    return cropped

def fine_crop_white_background(crop_img, debug_dir, namestem):
    """细裁剪：排除红色/粉色区域，保留白色背景矩形"""
    hsv = cv2.cvtColor(crop_img, cv2.COLOR_BGR2HSV)

    # 红色/粉色范围（小鼠躯体）
    mask_mouse1 = cv2.inRange(hsv, (0, 50, 50), (20, 255, 255))
    mask_mouse2 = cv2.inRange(hsv, (160, 50, 50), (180, 255, 255))
    mask_mouse = cv2.bitwise_or(mask_mouse1, mask_mouse2)

    # 反转，保留非红色区域
    mask_clean = cv2.bitwise_not(mask_mouse)
    cleaned = cv2.bitwise_and(crop_img, crop_img, mask=mask_clean)
    cv2.imwrite(str(Path(debug_dir) / f"{namestem}_fine_clean.jpg"), cleaned)

    # 找白色背景区域
    gray = cv2.cvtColor(cleaned, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
    cv2.imwrite(str(Path(debug_dir) / f"{namestem}_fine_binary.jpg"), binary)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print("未找到白色背景区域")
        return cleaned

    all_points = np.vstack(contours)
    x, y, w, h = cv2.boundingRect(all_points)
    fine_crop = cleaned[y:y+h, x:x+w]
    cv2.imwrite(str(Path(debug_dir) / f"{namestem}_fine_crop.jpg"), fine_crop)
    return fine_crop

def preprocess_for_ocr(crop, debug_dir, namestem):
    """灰度 → 自适应阈值 → 形态学清理"""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(str(Path(debug_dir) / f"{namestem}_gray.jpg"), gray)

    # 自适应阈值：动态区分文字和背景
    thresh = cv2.adaptiveThreshold(gray, 255,
                                   cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY_INV,
                                   25, 10)
    cv2.imwrite(str(Path(debug_dir) / f"{namestem}_thresh.jpg"), thresh)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    cv2.imwrite(str(Path(debug_dir) / f"{namestem}_morph.jpg"), morph)

    return morph

def parse_ms_dy_numbers(text: str):
    """
    从OCR文本中提取两个数字：
    - first_num：在 'dy' 之前的最后一个数字
    - second_num：在 'dy-' 之后的第一个数字
    """
    t = text.lower().replace(" ", "")
    t = t.replace("_", "-").replace("--", "-").replace("dγ", "dy").replace("d y", "dy")

    dy_idx = t.find("dy")
    if dy_idx == -1:
        return None, None

    # second_num：在 "dy-" 之后的第一个数字
    after = t[dy_idx+2:]
    if after.startswith("-"):
        after = after[1:]
    m2 = re.search(r'(\d+)', after)
    second_num = m2.group(1) if m2 else None

    # first_num：在 dy 之前的最后一个数字
    before = t[:dy_idx]
    m1 = re.findall(r'(\d+)', before)
    first_num = m1[-1] if m1 else None

    return first_num, second_num


def save_result_image(orig_img, rotated_img, need_rotate, save_path):
    if need_rotate:
        rotated_img.save(save_path, "JPEG", quality=95)
    else:
        orig_img.save(save_path, "JPEG", quality=95)

def save_ocr_image_with_index(img_path, output_dir, rename_failed_dir, debug_dir="debug"):
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    fail_dir = Path(rename_failed_dir); fail_dir.mkdir(parents=True, exist_ok=True)
    debug_dir = Path(debug_dir); debug_dir.mkdir(parents=True, exist_ok=True)

    namestem = Path(img_path).stem
    create_date = namestem[:6]

    csv_path = output_dir / "rename_process.csv"
    file_exists = csv_path.exists()
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(['original name', 'rename', 'raw ocr'])

    try:
        with Image.open(img_path) as orig_img:
            w, h = orig_img.size
            need_rotate = h > w
            if need_rotate:
                rotated_img = orig_img.rotate(90, expand=True)
                cv_img = cv2.cvtColor(np.array(rotated_img), cv2.COLOR_RGB2BGR)
            else:
                cv_img = cv2.cvtColor(np.array(orig_img.convert('RGB')), cv2.COLOR_RGB2BGR)

            # 1. 粗裁剪
            precrop = pre_crop_top_middle(cv_img, debug_dir, namestem)

            # 2. 细裁剪（排除红色/粉色）
            finecrop = fine_crop_white_background(precrop, debug_dir, namestem)

            # 3. 阈值化 + 清理
            processed = preprocess_for_ocr(finecrop, debug_dir, namestem)

            # 4. OCR
            config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=mdys0123456789-'
            raw_text = pytesseract.image_to_string(processed, config=config)
            #raw_text = pytesseract.image_to_string(morph, config=config)
            print("原始OCR文本:", raw_text)

            first_num, second_num = parse_ms_dy_numbers(raw_text)
            print(first_num,second_num)
            if first_num and second_num:
                base_name = f"ms-{first_num}-dy-{second_num}"
                matched = 1
            else:
                matched = 0

            # 5. 保存结果
            if matched == 1:
                counter = 1
                while True:
                    new_filename = f"{base_name}_{counter}_{create_date}.jpg"
                    save_path = output_dir / new_filename
                    if not save_path.exists():
                        save_result_image(orig_img, rotated_img if need_rotate else orig_img, need_rotate, save_path)
                        with open(csv_path, 'a', newline='', encoding='utf-8') as f:
                            csv.writer(f).writerow([os.path.basename(img_path), new_filename, raw_text])
                        print("保存成功:", save_path)
                        break
                    counter += 1
            else:
                new_filename = f"{namestem}_{create_date}.jpg"
                save_path = fail_dir / new_filename
                save_result_image(orig_img, rotated_img if need_rotate else orig_img, need_rotate, save_path)
                with open(csv_path, 'a', newline='', encoding='utf-8') as f:
                    csv.writer(f).writerow([os.path.basename(img_path), "failed", raw_text])
                print("识别失败，保存到:", save_path)

        return str(save_path)

    except Exception as e:
        print("处理失败:", str(e))
        return None


In [ ]:
import shutil
shutil.rmtree("/kaggle/working/rename/")
shutil.rmtree("/kaggle/working/rename-failed/")

In [ ]:
if __name__ == "__main__":
    test_result = save_ocr_image_with_index("/kaggle/input/250425-mouse-lnandgf/IMG_4572.JPG", 
                                            "/kaggle/working/rename/","/kaggle/working/rename-faled/")
    print(f"保存路径：{test_result}")

In [ ]:

input_dir = "/kaggle/input/set2ms/set2image/"
output_dir = "/kaggle/working/rename/"
rename_failed_dir="/kaggle/working/rename-failed/"

In [ ]:
from pathlib import Path


"""
批量处理文件夹中的JPG图片
:param input_dir: 输入文件夹路径
:param output_dir: 输出文件夹路径
"""
# 确保使用绝对路径
input_folder = Path(input_dir).resolve()
output_path = Path(output_dir).resolve()
rename_failed_outpath= Path(rename_failed_dir).resolve()


# 获取所有JPG文件路径（包括子目录）
image_paths = list(input_folder.rglob('*.jpg')) + list(input_folder.rglob('*.JPG'))

print(f"发现 {len(image_paths)} 张待处理图片")

# 批量处理
for idx, img_path in enumerate(image_paths, 1):
    print(f"\n正在处理第 {idx}/{len(image_paths)} 张: {img_path.name}")
    
    result = save_ocr_image_with_index(img_path,output_path,rename_failed_outpath)
    
    if result:
        print(f"成功保存至: {Path(result).name}")
    else:
        print(f"处理失败: {img_path.name}")



In [ ]:
import shutil
shutil.rmtree("/kaggle/working/rename/")